In [1]:
import sys
sys.path.append("../")

from operator import itemgetter

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import (
    RunnableLambda,
    RunnablePassthrough
)

from langchain_ollama import ChatOllama

c:\Users\gagan\Desktop\Langchain\labs\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from src.loaders.document_loader import EnterpriseDocumentLoader

from src.splitters.chunker import EnterpriseChunker

from src.embeddings.embedding_manager import EmbeddingManager

from vector_store.faiss_store import EnterpriseFAISS

from src.retriever.hybrid_retriever import EnterpriseHybridRetriever

from src.rerankers.reranker import EnterpriseReranker

from src.retriever.parent_retriever import EnterpriseParentRetriever

In [3]:
llm = ChatOllama(

    model="mistral",

    temperature=0
)

In [6]:
loader = EnterpriseDocumentLoader()

documents = loader.load_directory(
    "../datasets/military"
)

In [8]:
chunker = EnterpriseChunker(
    chunk_size=500,
    overlap=100
)

chunks = chunker.split_documents(documents)

In [9]:
embedding = EmbeddingManager(
    provider="ollama",
    model_name="nomic-embed-text"
)

In [11]:
faiss = EnterpriseFAISS(
    embedding
)

faiss.create(chunks)

In [12]:
hybrid = EnterpriseHybridRetriever(

    documents=chunks,

    vector_retriever=faiss.retriever(k=20),

    bm25_weight=0.4,

    vector_weight=0.6,

    k=20
)

In [13]:
reranker = EnterpriseReranker(

    retriever=hybrid.hybrid,

    top_n=5
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6194.52it/s]


In [14]:
docs = reranker.search(
    "Explain radar calibration."
)

In [15]:
def format_docs(docs):
    """
    Convert retrieved documents into a formatted context string.
    """

    formatted_docs = []

    for doc in docs:

        metadata = doc.metadata

        source = metadata.get("filename", "Unknown")
        page = metadata.get("page", "-")
        chunk = metadata.get("chunk_id", "-")

        formatted_docs.append(
            f"""
Source : {source}
Page   : {page}
Chunk  : {chunk}

Content:
{doc.page_content}
"""
        )

    return "\n\n" + "=" * 80 + "\n\n".join(formatted_docs)

In [16]:
RAG_PROMPT = ChatPromptTemplate.from_template(
"""
You are an Enterprise AI Assistant.

Rules:

1. Answer ONLY using the retrieved context.
2. Never use your own knowledge.
3. Never hallucinate.
4. If the answer is missing, reply exactly:

I don't have enough information in the provided documents.

Always provide:

Answer:

Sources:
- File
- Page
- Chunk

Confidence:
High / Medium / Low

Retrieved Context
-----------------

{context}

-----------------

Question:

{question}
"""
)

In [17]:
generation_chain = (

    {

        "context":

        RunnableLambda(
            lambda x: format_docs(x["docs"])
        ),

        "question":

        itemgetter("question")

    }

    | RAG_PROMPT

    | llm

    | StrOutputParser()

)

In [18]:
response = generation_chain.invoke(

    {

        "docs": docs,

        "question":"Explain radar calibration."

    }

)

print(response)

 I don't have enough information in the provided documents to explain radar calibration specifically. However, based on the context, it seems that the Radar Manual mentions the importance of staying vigilant and using radar discipline, but it does not provide details about calibration. The Weapons.csv file contains data about various weapons, including their categories, calibers, fire modes, effective ranges, etc., but there is no mention of radar calibration.

Sources:
- Radar_Manual.pdf (Page 1, Chunk 13)
- Weapons.csv (Multiple chunks)

Confidence: Low


In [19]:
def ask(question):

    docs = reranker.search(question)

    return generation_chain.invoke(

        {

            "docs":docs,

            "question":question

        }

    )

In [20]:
print(

    ask(

        "Explain radar calibration."

    )

)

 I don't have enough information in the provided documents to explain radar calibration specifically. However, based on the context, it seems that the Radar Manual mentions the importance of staying vigilant and using radar discipline, but it does not provide details about calibration. The Weapons.csv file contains data about various weapons, including their categories, calibers, fire modes, effective ranges, etc., but there is no mention of radar calibration.

Sources:
- Radar_Manual.pdf (Page 1, Chunk 13)
- Weapons.csv (Multiple chunks)

Confidence: Low
